# Download COCO Datasets

# images — ~778 MB
!wget -c http://images.cocodataset.org/zips/val2017.zip

# annotations — ~241 MB (contains train + val; you need the val one)
!wget -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip val2017.zip
!unzip annotations_trainval2017.zip



# Testing on YOLO World vs COCO dataset for object detection and comparison of results.

In [4]:
import os, torch
from pycocotools.coco import COCO
from ultralytics import YOLOWorld
from torchmetrics.detection import MeanAveragePrecision

In [5]:
ANN     = "coco/annotations/instances_val2017.json"
IMG_DIR = "coco/val2017"
PERSON  = 1                      # COCO category id for person


In [ ]:
coco    = COCO(ANN)
img_ids = sorted(coco.getImgIds())[:100]


loading annotations into memory...
Done (t=1.68s)
creating index...
index created!


In [7]:
model = YOLOWorld("yolov8s-world.pt")     # first run downloads ~50 MB
model.set_classes(["person"])

records = []                              # keep raw dets for offline sweeping


In [8]:
for iid in img_ids:
    info = coco.loadImgs(iid)[0]
    path = os.path.join(IMG_DIR, info["file_name"])

    anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=[PERSON], iscrowd=False))
    gt = torch.tensor( 
        [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
        dtype=torch.float32,
    ).reshape(-1, 4)
                                                               
    r = model.predict(path, conf=0.001, max_det=300, verbose=False)[0]
    records.append({"boxes": r.boxes.xyxy.cpu(), "scores": r.boxes.conf.cpu(), "gt": gt})

print(f"{len(records)} images, "
      f"{sum(len(x['gt']) for x in records)} ground-truth people, "
      f"{sum(len(x['boxes']) for x in records)} raw detections")


100 images, 283 ground-truth people, 3661 raw detections


In [9]:
def evaluate(records, thresh):
    metric = MeanAveragePrecision(iou_type="bbox")
    for r in records:
        keep = r["scores"] >= thresh
        metric.update(
            [{"boxes": r["boxes"][keep],
              "scores": r["scores"][keep],
              "labels": torch.zeros(int(keep.sum()), dtype=torch.long)}],
            [{"boxes": r["gt"],
              "labels": torch.zeros(len(r["gt"]), dtype=torch.long)}],
        )
    m = metric.compute()
    return float(m["map_50"]), float(m["map"])


In [10]:
for t in [0.01, 0.05, 0.10, 0.25, 0.40, 0.60]:
    ap50, ap = evaluate(records, t)
    print(f"conf {t:.2f}   mAP50 {ap50:.3f}   mAP50-95 {ap:.3f}")


conf 0.01   mAP50 0.713   mAP50-95 0.506
conf 0.05   mAP50 0.692   mAP50-95 0.493
conf 0.10   mAP50 0.651   mAP50-95 0.469
conf 0.25   mAP50 0.552   mAP50-95 0.420
conf 0.40   mAP50 0.483   mAP50-95 0.372
conf 0.60   mAP50 0.373   mAP50-95 0.308
